# ASL Sign Model — Colab Trainer
From-scratch tiny CNN + temporal head, trained on the preprocessed ASL Citizen 75-sign cache.

**Before running:** set `Runtime -> Change runtime type -> T4 GPU` (free tier is enough).

**One-time setup:** locally run `python -m asl.package_for_colab`, then upload
`artifacts/colab_bundle.zip` to Google Drive at `MyDrive/asl-model/colab_bundle.zip`.

All real logic lives in the `asl` package (identical to local runs); this notebook just runs it.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, zipfile
BUNDLE = '/content/drive/MyDrive/asl-model/colab_bundle.zip'
os.makedirs('/content/work', exist_ok=True)
with zipfile.ZipFile(BUNDLE) as z:
    z.extractall('/content/work')
%cd /content/work
!pip -q install pyyaml onnx onnxruntime
import torch; print('cuda available:', torch.cuda.is_available())

In [ ]:
# Train (CUDA is auto-selected by asl.train.device()). Pooling baseline:
!PYTHONPATH=src python -m asl.train --config configs/baseline.yaml
# To try the Transformer head instead:
# !PYTHONPATH=src python -m asl.train --config configs/baseline.yaml --head transformer

In [ ]:
# Export to ONNX + meta.json and copy results back to Drive
!PYTHONPATH=src python -m asl.export --ckpt artifacts/checkpoints/baseline/best.pt --out-dir export --version v0.1 --quantize
import shutil, os
os.makedirs('/content/drive/MyDrive/asl-model/out', exist_ok=True)
for f in ['export', 'artifacts/checkpoints/baseline/best.pt', 'artifacts/checkpoints/baseline/history.json']:
    dst = '/content/drive/MyDrive/asl-model/out/' + os.path.basename(f.rstrip('/'))
    (shutil.copytree if os.path.isdir(f) else shutil.copy)(f, dst, dirs_exist_ok=True) if os.path.isdir(f) else shutil.copy(f, dst)
print('Results copied to MyDrive/asl-model/out')